# Read SDTM CT — the base graph of SDTM Controlled Terminology (scoped)

The pinned NCI EVS file `SDTM Terminology.txt`, as published, for the codelists in scope:

1. pin check (`Reference_Versions.json`);
2. the file as a text table (`cfa.read.read_tsv_table`: tab-separated, no quote handling); the header must equal the schema's columns;
3. **the scope** (the class annotation `scope` in `schema/sdtm_ct.yaml`), read from the base graphs of COSMoS (`base/cosmos/cosmos.ttl`) and SDTMIG (`base/sdtmig/sdtmig.ttl`, built locally: sign-in content): the codelist row and every term row of each codelist in scope. Codelists referred to but without a codelist row in this file are listed, not an error;
4. the graph of the rows in scope by the CSVW table description generated from the module (`cfa.rule.table_to_graph`);
5. structural SHACL: must conform;
6. writes `sdtm-ct.ttl` (longturtle) and its graph hash;
7. the Excel view `SDTM_Terminology.xlsx`, rebuilt from the graph as written; its text hash must equal the text hash of the source rows in scope;
8. `Provenance.json`, with the scope rule and the graphs it read.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph, URIRef

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.pins import checked_source_file, table_file
from cfa.provenance import flat, provenance, write_provenance
from cfa.read import read_tsv_table, read_xlsx_table
from cfa.rule import (check_module, csvw_metadata, graph_iri, key_columns, load_module, module_source,
                      pct, property_iri, table_to_graph)
from cfa.view import schema_sheet, text_sha256, view_frame, write_view

MODULE = SCHEMA_DIR / 'sdtm_ct.yaml'
TABLE = 'SDTM Terminology'
sv = load_module(MODULE)
check_module(sv)
SOURCE = module_source(sv)
OUT = REPO_ROOT / 'base' / SOURCE
KEYS = key_columns(sv, TABLE)
SCOPE_RULE = sv.get_class(TABLE).annotations['scope'].value
print(SOURCE, TABLE, KEYS)
print('scope:', SCOPE_RULE)

## The generated contract is current

In [ ]:
CSVW_PATH = SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json'
SHACL_PATH = SCHEMA_DIR / 'shacl' / f'{SOURCE}.shacl.ttl'
meta = csvw_metadata(sv, TABLE, f'{pct(TABLE)}.csv')
assert json.loads(CSVW_PATH.read_text(encoding='utf-8')) == meta, 'CSVW not current: run checks/00_Schema'
assert SHACL_PATH.read_text(encoding='utf-8') == ShaclGenerator(str(MODULE)).serialize(), 'SHACL not current: run checks/00_Schema'
print('contract current')

## 1–2. Pin check, the file as a text table

In [ ]:
src = checked_source_file(SOURCE, table_file(SOURCE, TABLE))
table = read_tsv_table(src)
is_codelist_row = table['Codelist Code'] == ''
ct_codelists = set(table.loc[is_codelist_row, 'Code'])
assert is_codelist_row.sum() == len(ct_codelists), 'a codelist code occurs on more than one codelist row'
print(src.name, '| rows', len(table), '| columns', len(table.columns))

## 3. The scope, from the COSMoS and SDTMIG base graphs

The graphs are read as written by their readers; their graph hashes go into Provenance.

In [ ]:
COSMOS_TTL = REPO_ROOT / 'base' / 'cosmos' / 'cosmos.ttl'
SDTMIG_TTL = REPO_ROOT / 'base' / 'sdtmig' / 'sdtmig.ttl'
cosmos = Graph().parse(COSMOS_TTL)
sdtmig = Graph().parse(SDTMIG_TTL)
p_cosmos = URIRef(property_iri('cosmos', 'cdisc_sdtm_dataset_specializations_latest', 'codelist'))
p_ig = URIRef(property_iri('sdtmig', 'SDTMIG', 'CDISC CT Codelist Code(s)'))
cosmos_refs = {str(o) for o in cosmos.objects(None, p_cosmos)}
ig_refs = {c.strip() for o in sdtmig.objects(None, p_ig) for c in str(o).split(';') if c.strip()}
scope = (cosmos_refs | ig_refs) & ct_codelists
NOT_IN_SOURCE = {'cosmos': sorted(cosmos_refs - ct_codelists), 'sdtmig': sorted(ig_refs - ct_codelists)}
in_scope = (is_codelist_row & table['Code'].isin(scope)) | table['Codelist Code'].isin(scope)
scoped = table[in_scope].reset_index(drop=True)
assert set(scoped.loc[scoped['Codelist Code'] == '', 'Code']) == scope
print(f'codelists referred to: COSMoS {len(cosmos_refs)}, SDTMIG {len(ig_refs)}; in scope {len(scope)} of {len(ct_codelists)}')
print('referred to, no codelist row in this file:', NOT_IN_SOURCE)
print('rows in scope:', len(scoped))

## 4–5. The graph, structural SHACL

In [ ]:
g = table_to_graph(meta, scoped)
conforms, _, report = pyshacl.validate(g, shacl_graph=Graph().parse(SHACL_PATH))
assert conforms, report
print('triples', len(g), '| SHACL conforms')

## 6. Write the graph, its hash

In [ ]:
TTL = OUT / f'{SOURCE}.ttl'
write_longturtle(g, TTL, {'ct': f'https://kerfors.github.io/cdisc-for-all/def/{SOURCE}/{pct(TABLE)}#'})
written = Graph().parse(TTL)
GRAPH_HASH = graph_sha256(written)
assert GRAPH_HASH == graph_sha256(g)
print(TTL.name, GRAPH_HASH)

## 7. The Excel view, lossless over the rows in scope

In [ ]:
view = view_frame(written, sv, TABLE)
SOURCE_HASH = text_sha256(scoped, KEYS)
VIEW_HASH = text_sha256(view, KEYS)
assert VIEW_HASH == SOURCE_HASH, 'the view differs from the source rows in scope'
print('text hash, source rows in scope = view:', VIEW_HASH)

## 8. Provenance; write the view and read it back

In [ ]:
prov = provenance(SOURCE, [src], [MODULE, CSVW_PATH, SHACL_PATH], graph_iri(SOURCE), len(written),
                  GRAPH_HASH, SOURCE_HASH, VIEW_HASH)
prov['scope'] = {
    'rule': SCOPE_RULE,
    'graphs_read': {'cosmos': graph_sha256(cosmos), 'sdtmig': graph_sha256(sdtmig)},
    'referred_to_without_codelist_row': NOT_IN_SOURCE,
}
write_provenance(OUT / 'Provenance.json', prov)
XLSX = OUT / 'SDTM_Terminology.xlsx'
readme = [
    'SDTM Controlled Terminology (NCI EVS file SDTM Terminology.txt), as published, for the codelists in scope.',
    f'Scope: {SCOPE_RULE}',
    f'Rebuilt from the graph {TTL.name}: columns in published order, rows sorted by the key ({", ".join(KEYS)}).',
    'Every cell is text. The Schema sheet gives each column\'s property IRI; Provenance gives the pin, the scope and the hashes.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
write_view(XLSX, readme, {TABLE: view}, schema_sheet(sv, [TABLE]), flat(prov))
assert text_sha256(read_xlsx_table(XLSX, TABLE), KEYS) == SOURCE_HASH
print(XLSX.name, 'read back: text hash equal')